<a href="https://colab.research.google.com/github/chuanbinp/uncertainty-aware-inference/blob/master/TeamB/teamb_vllm_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Team B — vLLM Throughput Benchmark
## Mistral-7B PTQ Sweep

Benchmarks **vLLM serving throughput** across all five quantization configs.
NF4 falls back to HF batched generation (vLLM does not support bitsandbytes).

| Config | Quantization | Kernel |
|--------|-------------|--------|
| `mistral-7b-fp16` | FP16 | cuBLAS |
| `mistral-7b-gptq-int4` | GPTQ INT4 | gptq_marlin |
| `mistral-7b-gptq-int8` | GPTQ INT8 | gptq (exllama) |
| `mistral-7b-awq-int4` | AWQ INT4 | awq_marlin |
| `mistral-7b-nf4` | NF4 bitsandbytes | HF batched fallback |

**To adapt for your model:** update `ALL_CONFIGS` and `REPO_DIR` in Section 3,
and add your model entries to `VLLM_CONFIGS` in `run_vllm.py`.


<a href="https://colab.research.google.com/github/chuanbinp/uncertainty-aware-inference/blob/master/TeamB/teamb_vllm_profiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup

### 1.1 Verify GPU

In [ ]:
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM            : {total_mem:.1f} GB")
    if total_mem < 20:
        print("WARNING: <20 GB VRAM — some configs may OOM. Recommend A100.")


### 1.2 Clone repo

In [ ]:
!git clone https://github.com/chuanbinp/uncertainty-aware-inference.git

### 1.3 Install core dependencies

In [ ]:
# Same package set as mistral_7b.ipynb — versions not changed
!pip install numpy plotly matplotlib torch transformers datasets accelerate gptqmodel netcal autoawq bitsandbytes


### 1.4 Install vLLM

vLLM is installed separately after core deps. `vllm==0.4.3` is compatible with the torch version above.

In [ ]:
!pip install vllm==0.4.3

### 1.5 Install W&B

In [ ]:
!pip install wandb

### 1.6 Verify scripts are present

In [ ]:
import os
REPO_DIR = "/content/uncertainty-aware-inference/TeamB"
for script in ["run_vllm.py", "run_profiler.py", "run_eval.py", "configs.py"]:
    path = os.path.join(REPO_DIR, script)
    status = "OK" if os.path.exists(path) else "MISSING — upload manually"
    print(f"  {script}: {status}")


# 2. Authentication

### 2.1 HuggingFace — required for gated Mistral-7B weights

In [ ]:
from huggingface_hub import login
from getpass import getpass
import os

HF_TOKEN = getpass("Enter your HuggingFace token: ")
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN   # propagate to subprocesses
print("HuggingFace login successful")


### 2.2 Weights & Biases

In [ ]:
import wandb
from getpass import getpass

WANDB_API_KEY = getpass("Enter your W&B API key (Enter to skip): ")
if WANDB_API_KEY.strip():
    wandb.login(key=WANDB_API_KEY)
    WANDB_ENABLED = True
    print("W&B login successful")
else:
    WANDB_ENABLED = False
    print("W&B disabled — results will be saved to JSON only")


# 3. Experiment Configuration

In [ ]:
import os
from pathlib import Path

REPO_DIR     = "/content/uncertainty-aware-inference/TeamB"
VLLM_OUT = "/content/vllm_results"

for d in [VLLM_OUT]:
    os.makedirs(d, exist_ok=True)

# All configs to sweep — same ordering as configs.py MODEL_REGISTRY
ALL_CONFIGS = [
    "mistral-7b-fp16",
    "mistral-7b-gptq-int4",
    "mistral-7b-gptq-int8",
    "mistral-7b-awq-int4",
    "mistral-7b-nf4",
]

WANDB_PROJECT = "UAI_Project"
WANDB_ENTITY  = "Uncertainty_Aware_Inference_Lab"

print(f"Sweep configs : {ALL_CONFIGS}")
print(f"vLLM output   : {VLLM_OUT}")
print(f"W&B enabled   : {WANDB_ENABLED}")


# 4. Run Helpers

In [ ]:
import subprocess, sys, json, os
from pathlib import Path

def run_script(script_path: str, extra_args: list):
    """Run a Python script as a subprocess, streaming stdout + stderr."""
    cmd = [sys.executable, script_path] + extra_args
    print(f"\nCMD: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        # Print last 3000 chars of stderr (avoids flooding for large models)
        stderr_tail = result.stderr[-3000:]
        print("STDERR (tail):", stderr_tail)
    return result.returncode == 0

def load_json(path: str) -> dict | None:
    p = Path(path)
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

def wandb_run_init(config_key: str, experiment: str):
    """Create and return a W&B run if enabled, else None."""
    if not WANDB_ENABLED:
        return None
    return wandb.init(
        project  = WANDB_PROJECT,
        entity   = WANDB_ENTITY,
        name     = f"teamB_{config_key}_{experiment}",
        reinit   = True,
        config   = {"model": "mistral-7b", "team": "team-b",
                    "config_key": config_key, "experiment": experiment},
    )

print("Helpers ready.")


# 5. vLLM Throughput Benchmark

Calls `run_vllm.py` for each config. The script:
- Uses a fresh vLLM `LLM` instance per run (subprocess = clean GPU state)
- Runs 2 warmup passes then 3 timed passes over 10 benchmark prompts (128 output tokens each)
- NF4 automatically uses HF batched generation (vLLM does not support bitsandbytes)
- Saves `{config_key}_vllm.json` to `VLLM_OUT`

In [ ]:
import json
from pathlib import Path

SCRIPT_VLLM = os.path.join(REPO_DIR, "run_vllm.py")
vllm_summary = {}

for config_key in ALL_CONFIGS:
    print(f"\n{'#'*65}")
    print(f"# vLLM benchmark: {config_key}")
    print(f"{'#'*65}")

    run = wandb_run_init(config_key, "vllm_throughput")
    run_id = run.id if run else None

    args = [
        "--config",     config_key,
        "--output-dir", VLLM_OUT,
        "--hf-token",   HF_TOKEN,
    ]
    if run_id:
        args += ["--wandb-run-id", run_id, "--wandb-project", WANDB_PROJECT]

    ok = run_script(SCRIPT_VLLM, args)

    result = load_json(f"{VLLM_OUT}/{config_key}_vllm.json")
    if result:
        vllm_summary[config_key] = result
        print(f"DONE {config_key}: {result['avg_tok_per_sec']:.1f} tok/s | kernel={result['kernel']}")
        if run:
            run.log({
                "vllm/avg_tok_per_sec": result["avg_tok_per_sec"],
                "vllm/peak_gpu_mem_gb": result["peak_gpu_mem_gb"],
                "vllm/kernel":          result["kernel"],
            })
    else:
        print(f"WARN: no JSON output for {config_key} — see stderr above")

    if run:
        run.finish()

print("\nvLLM benchmark complete for all configs.")


### 5.1 vLLM Results Summary

In [ ]:
import json
from pathlib import Path

COLS = ["Config", "Avg tok/s", "Peak GPU GB", "Kernel"]
rows = []
for cfg in ALL_CONFIGS:
    d = load_json(f"{VLLM_OUT}/{cfg}_vllm.json")
    if d:
        rows.append([cfg, f"{d['avg_tok_per_sec']:.2f}", f"{d['peak_gpu_mem_gb']:.3f}", d["kernel"]])
    else:
        rows.append([cfg, "N/A", "N/A", "N/A"])

w = [30, 12, 12, 35]
header = "  ".join(f"{c:<{w[i]}}" for i, c in enumerate(COLS))
print(header)
print("-" * sum(w))
for r in rows:
    print("  ".join(f"{v:<{w[i]}}" for i, v in enumerate(r)))


# 6. Download Results

In [ ]:
!zip -r /content/vllm_results.zip /content/vllm_results/
print("Zipped vllm_results.zip")


In [ ]:
from google.colab import files
files.download("/content/vllm_results.zip")
